# 8‑VSB Simulator (Vestigial Sideband) — Complete Jupyter Notebook

این نوت‌بوک یک شبیه‌ساز کامل پایه‌ای برای **8‑VSB** (Vestigial Sideband) ارائه می‌دهد.
قابلیت‌ها:
- نگاشت بیت‌ها به سمبل‌های 8‑VSB (8‑level PAM)
- فیلتر شکل‌دهی پالسی (Raised-Cosine / root-raised-cosine-like)
- اعمال کانال AWGN (و امکان اعمال تاخیر/چرخش فاز)
- فیلتر تطبیقی (LMS) برای برطرف‌سازی اعوجاج کانال
- بازیابی زمان سمبل با الگوریتم Gardner (پیاده‌سازی ساده)
- دمدولاسیون و محاسبهٔ BER
- نمودارهای مفید (صورت فضایی، زمان، BER vs SNR)

نحوهٔ استفاده: سلول‌ها را از بالا به پایین اجرا کنید. پارامترها (نمونه‌گیری، نرخ سمبل، SNR، تعداد آزمون‌ها) قابل تنظیم هستند.


In [ ]:

# -*- coding: utf-8 -*-
# 8-VSB simulator cell
import numpy as np
import matplotlib.pyplot as plt
import random
from typing import Tuple, List

# ----------------------------
# Parameters (قابل تنظیم)
# ----------------------------
M = 8  # 8-VSB => 8 amplitude levels (PAM-8)
bits_per_sym = 3
fs = 24000  # sampling frequency in Hz (for simulation)
sym_rate = 1000  # symbol rate in symbols/sec
oversample = int(fs / sym_rate)  # samples per symbol
if oversample < 4:
    oversample = 4
    fs = sym_rate * oversample

# filter parameters
rolloff = 0.25
span_symbols = 10  # filter span in symbols

# Simulation control
num_symbols = 20000  # number of symbols per frame
snr_db = 20.0  # SNR in dB (for AWGN)
seed = 12345

np.random.seed(seed)
random.seed(seed)

# ----------------------------
# Utility functions
# ----------------------------
def bits_to_symbols(bits: List[int]) -> np.ndarray:
    '''Map bits to 8-VSB levels. Gray mapping recommended.'''
    bits = np.array(bits).reshape(-1, 3)
    ints = bits[:,0]*4 + bits[:,1]*2 + bits[:,2]*1
    levels = 2*ints - (M-1)
    return levels.astype(float)

def symbols_to_bits(symbols: np.ndarray) -> np.ndarray:
    '''Demap 8-PAM levels back to bits (simple nearest decision)'''
    levels = np.array([-7,-5,-3,-1,1,3,5,7])
    idx = np.argmin(np.abs(symbols.reshape(-1,1) - levels.reshape(1,-1)), axis=1)
    bits = np.vstack([ (idx & 4) >> 2, (idx & 2) >> 1, idx & 1 ]).T.flatten()
    return bits.astype(int)

def awgn(signal: np.ndarray, snr_db: float) -> np.ndarray:
    '''Add AWGN to a real signal for a given SNR in dB (SNR per symbol).'''
    sig_power = np.mean(signal**2)
    snr_linear = 10**(snr_db/10.0)
    noise_power = sig_power / snr_linear
    noise = np.sqrt(noise_power) * np.random.randn(*signal.shape)
    return signal + noise

# ----------------------------
# Pulse shaping: Raised cosine (approx root-raised by using matched filter)
# ----------------------------
def raised_cosine_filter(rolloff: float, span: int, sps: int) -> np.ndarray:
    N = span * sps
    t = np.arange(-N/2, N/2 + 1) / float(sps)
    h = np.zeros_like(t)
    for i, ti in enumerate(t):
        if np.isclose(ti, 0.0):
            h[i] = 1.0 - rolloff + 4*rolloff/np.pi
        elif np.isclose(abs(ti), 1/(4*rolloff)) and rolloff != 0:
            h[i] = (rolloff/np.sqrt(2.0)) * ((1+2/np.pi) * np.sin(np.pi/(4*rolloff)) + (1-2/np.pi)*np.cos(np.pi/(4*rolloff)))
        else:
            num = np.sin(np.pi*ti*(1-rolloff)) + 4*rolloff*ti*np.cos(np.pi*ti*(1+rolloff))
            den = np.pi*ti*(1-(4*rolloff*ti)**2)
            h[i] = num/den
    h = h / np.sqrt(np.sum(h**2))
    return h

# ----------------------------
# Transmitter
# ----------------------------
def transmitter(bits: np.ndarray, sps: int, rolloff: float, span: int) -> Tuple[np.ndarray, np.ndarray]:
    symbols = bits_to_symbols(bits)
    up = np.zeros(len(symbols) * sps)
    up[::sps] = symbols
    h = raised_cosine_filter(rolloff, span, sps)
    tx = np.convolve(up, h, mode='same')
    return tx, h

# ----------------------------
# Receiver (matched filter + timing + equalizer)
# ----------------------------
def matched_filter_and_sample(rx: np.ndarray, h: np.ndarray, sps: int, timing_offset: float=0.0) -> np.ndarray:
    mf = h[::-1].copy()
    y = np.convolve(rx, mf, mode='same')
    n_syms = int(len(y) / sps)
    indices = (np.arange(n_syms) * sps + int(sps/2) ) + timing_offset
    indices = indices.astype(float)
    samples = np.interp(indices, np.arange(len(y)), y)
    return samples

def lms_equalizer(train_symbols: np.ndarray, rx_samples: np.ndarray, sps: int, eq_taps: int=11, mu: float=0.001) -> Tuple[np.ndarray, np.ndarray]:
    N = len(rx_samples)
    w = np.zeros(eq_taps)
    w[eq_taps//2] = 1.0
    pad = np.zeros(eq_taps-1)
    rx_padded = np.concatenate([pad, rx_samples, pad])
    out = np.zeros_like(rx_samples)
    for n in range(len(rx_samples)):
        x = rx_padded[n:n+eq_taps][::-1]
        y = np.dot(w, x)
        d = train_symbols[n] if n < len(train_symbols) else np.round(y/2)*2
        e = d - y
        w += mu * e * x
        out[n] = y
    return out, w

def gardner_timing_recovery(rx: np.ndarray, sps: int, n_symbols: int, mu0: float=0.0, gain: float=0.01) -> Tuple[np.ndarray, float]:
    '''A simplified Gardner loop that estimates fractional timing offset mu and returns sampled symbols.'''
    mu = mu0
    t = 0.0
    samples = []
    for k in range(n_symbols):
        samp = np.interp(t + mu, np.arange(len(rx)), rx)
        samp_early = np.interp(t - 0.5*sps, np.arange(len(rx)), rx)
        samp_late = np.interp(t + 0.5*sps, np.arange(len(rx)), rx)
        samples.append(samp)
        e = (samp_early - samp_late) * samp
        mu += gain * e
        t += sps
    return np.array(samples), mu

# ----------------------------
# Full simulation run
# ----------------------------
def run_simulation(num_symbols: int, sps: int, rolloff: float, span: int, snr_db: float):
    bits = np.random.randint(0, 2, size=num_symbols * bits_per_sym)
    tx_signal, h = transmitter(bits, sps, rolloff, span)
    timing_offset = np.random.uniform(-0.3*sps, 0.3*sps)
    rx = awgn(tx_signal, snr_db)
    samples = matched_filter_and_sample(rx, h, sps, timing_offset=timing_offset)
    train_len = min(1000, len(samples)//2)
    train_symbols = bits_to_symbols(bits[:train_len*bits_per_sym])
    eq_out, taps = lms_equalizer(train_symbols, samples, sps, eq_taps=11, mu=0.001)
    decided_bits = symbols_to_bits(eq_out[:num_symbols])
    tx_bits = bits[:len(decided_bits)]
    bit_errors = np.sum(tx_bits != decided_bits[:len(tx_bits)])
    ber = bit_errors / len(tx_bits)
    return {
        'bits': bits,
        'tx_signal': tx_signal,
        'rx_signal': rx,
        'h': h,
        'samples': samples,
        'eq_out': eq_out,
        'taps': taps,
        'ber': ber
    }

# Run an example simulation
res = run_simulation(num_symbols, oversample, rolloff, span_symbols, snr_db)

# Print summary
print("Simulation completed.")
print("SNR (dB):", snr_db)
print("Estimated BER:", res['ber'])

# ----------------------------
# Plots: waveform, filter impulse, sampled and equalized symbols
# ----------------------------
plt.figure()
plt.plot(res['tx_signal'][:500])
plt.title('Transmitted waveform (first 500 samples)')
plt.xlabel('sample index')
plt.ylabel('amplitude')

plt.figure()
plt.plot(res['rx_signal'][:500])
plt.title('Received waveform (first 500 samples)')
plt.xlabel('sample index')
plt.ylabel('amplitude')

plt.figure()
plt.stem(res['h'])
plt.title('Pulse shaping filter impulse (raised cosine approximate)')
plt.xlabel('tap')
plt.ylabel('value')

plt.figure()
plt.plot(res['samples'][:200], marker='o', linestyle='None')
plt.title('Sampled symbols (first 200)')
plt.xlabel('symbol index')
plt.ylabel('amplitude')

plt.figure()
plt.plot(res['eq_out'][:200], marker='o', linestyle='None')
plt.title('Equalized output (first 200 symbols)')
plt.xlabel('symbol index')
plt.ylabel('amplitude')

plt.show()
